Missouri Manual Semantic Categories:
•Agriculture
•Cities
•Community engagement
•Cost of living – Services – Healthcare
•Culture
•Diversity
•Economy/Commerce/Industry
•Environment
•Ideology
•Infrastructure
•Elderly
•Environment
•Family – Children
•K12
•Named neighborhood
•NIMBY
•Policing
•Poverty
•Recreation – Tourism
•Religion
•Suburbs
•Technology
•University
•Violence
•Vulnerable populations

In [ ]:
# pip install transformers spacy torch torchvision
# python -m spacy download en_core_web_sm

import pandas as pd
import spacy
from transformers import pipeline

nlp_spacy = spacy.load("en_core_web_sm")
def remove_geography(text):
    if not isinstance(text, str):
        return ""
    doc = nlp_spacy(text)
    # remove geopolitical entities and locations
    # why? mggg models were overfitting to geography
    clean_text = " ".join([token.text for token in doc if token.ent_type_ not in ['GPE', 'LOC']])
    return clean_text



# load data
df = pd.read_csv("data/MOCumulativeAug10.csv")
df['clean_text'] = df['text'].apply(remove_geography)


# load classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# agggregating communities paper labels
candidate_labels = [
    "Agriculture", "Cities", "Community engagement", "Cost of living", 
    "Culture", "Diversity", "Economy and Commerce", "Environment", 
    "Ideology", "Infrastructure", "Elderly", "Family and Children", 
    "K-12 Education", "Named neighborhood", "NIMBY", "Policing", 
    "Poverty", "Recreation and Tourism", "Religion", "Suburbs", 
    "Technology", "University", "Violence", "Vulnerable populations"
]

def categorize_comment(text):
    if len(text.strip()) < 5:
        return "Unknown"
    result = classifier(text, candidate_labels, multi_label=False)
    
    # top probability label
    return result['labels'][0]



df['predicted_category'] = df['clean_text'].apply(categorize_comment)
df.to_csv('MOCategorizedComments')

'''
for index, row in df.iterrows():
    print(f"\noriginal text: {row['text']}...")
    print(f"predicted category: {row['predicted_category']}")

'''

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

'\nfor index, row in df.iterrows():\n    print(f"\noriginal text: {row[\'text\']}...")\n    print(f"predicted category: {row[\'predicted_category\']}")\n\n'